# 02 — Exploratory Data Analysis\n**ENAI 603 | WMATA Metro Delay Prediction**\n\nThis notebook explores the engineered feature set produced by `scripts/build_features.py`.\n\n**Outline**\n1. Dataset overview & missing values\n2. Target variable (delay) distribution\n3. Temporal patterns\n4. Line & station analysis\n5. Real-time & rolling features\n6. Incident impact\n7. Feature correlations\n8. Key takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)

PROJ = os.path.abspath(os.path.join(os.getcwd(), ".."))
FIG_DIR = os.path.join(PROJ, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROJ, "data", "features.csv"), parse_dates=["collected_at"])
print(f"Shape: {df.shape}")
df.head()

## 1. Dataset Overview & Missing Values

In [ ]:
print(df.dtypes.to_string())
print(f"\nRows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"\nDate range: {df['collected_at'].min()} → {df['collected_at'].max()}")

In [ ]:
# Missing values
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
if len(nulls) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    nulls.plot.barh(ax=ax, color="salmon")
    ax.set_xlabel("Missing count")
    ax.set_title("Missing Values by Feature")
    for i, v in enumerate(nulls):
        ax.text(v + 200, i, f"{v:,} ({v/len(df):.0%})", va="center")
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "eda_missing_values.png"), dpi=150)
    plt.show()
else:
    print("No missing values!")

## 2. Target Variable — Delay Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts = df["is_delayed"].value_counts().sort_index()
labels = ["On-time (0)", "Delayed (1)"]
colors = ["#4CAF50", "#F44336"]
axes[0].bar(labels, counts.values, color=colors)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}\n({v/len(df):.1%})", ha="center", fontweight="bold")
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")

# Delay rate by line
delay_by_line = df.groupby("line")["is_delayed"].mean().sort_values(ascending=True)
line_colors = {"RD": "#C80F22", "BL": "#009CDE", "GR": "#00B140",
               "YL": "#FFD100", "OR": "#F7941D", "SV": "#A2A4A1"}
bar_colors = [line_colors.get(l, "gray") for l in delay_by_line.index]
delay_by_line.plot.barh(ax=axes[1], color=bar_colors)
axes[1].set_xlabel("Delay Rate")
axes[1].set_title("Delay Rate by Line")
for i, v in enumerate(delay_by_line.values):
    axes[1].text(v + 0.005, i, f"{v:.1%}", va="center")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_delay_distribution.png"), dpi=150)
plt.show()

## 3. Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Delay rate by hour
hourly = df.groupby("hour")["is_delayed"].agg(["mean", "count"]).reset_index()
axes[0].bar(hourly["hour"], hourly["mean"], color="steelblue", alpha=0.8)
axes[0].axhline(df["is_delayed"].mean(), color="red", ls="--", label=f'Overall: {df["is_delayed"].mean():.1%}')
axes[0].set_xlabel("Hour of Day (ET)")
axes[0].set_ylabel("Delay Rate")
axes[0].set_title("Delay Rate by Hour")
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

# Volume by hour
axes[1].bar(hourly["hour"], hourly["count"], color="gray", alpha=0.7)
axes[1].set_xlabel("Hour of Day (ET)")
axes[1].set_ylabel("Prediction Count")
axes[1].set_title("Data Volume by Hour")
axes[1].set_xticks(range(0, 24, 2))

# Rush hour vs off-peak
rush_delay = df.groupby("is_rush_hour")["is_delayed"].mean()
rush_labels = ["Off-Peak", "Rush Hour"]
axes[2].bar(rush_labels, rush_delay.values, color=["#78909C", "#E53935"])
for i, v in enumerate(rush_delay.values):
    axes[2].text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")
axes[2].set_ylabel("Delay Rate")
axes[2].set_title("Rush Hour vs Off-Peak")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_temporal_patterns.png"), dpi=150)
plt.show()

## 4. Line & Station Analysis

In [ ]:
# Top 15 stations by delay rate (min 100 observations)
station_stats = df.groupby(["location_code", "location_name"]).agg(
    delay_rate=("is_delayed", "mean"),
    count=("is_delayed", "count")
).reset_index()
station_stats = station_stats[station_stats["count"] >= 100].sort_values("delay_rate", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 15 most delayed
top15 = station_stats.tail(15)
axes[0].barh(top15["location_name"], top15["delay_rate"], color="tomato")
axes[0].set_xlabel("Delay Rate")
axes[0].set_title("Top 15 Most Delayed Stations")
for i, v in enumerate(top15["delay_rate"].values):
    axes[0].text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)

# Bottom 15 (least delayed)
bot15 = station_stats.head(15)
axes[1].barh(bot15["location_name"], bot15["delay_rate"], color="mediumseagreen")
axes[1].set_xlabel("Delay Rate")
axes[1].set_title("Top 15 Least Delayed Stations")
for i, v in enumerate(bot15["delay_rate"].values):
    axes[1].text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_station_delays.png"), dpi=150)
plt.show()

In [ ]:
# Delay rate heatmap: line × hour
pivot = df.groupby(["line", "hour"])["is_delayed"].mean().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt=".0%", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "Delay Rate"}, linewidths=0.5)
ax.set_title("Delay Rate: Line × Hour of Day")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_line_hour_heatmap.png"), dpi=150)
plt.show()

## 5. Real-Time & Rolling Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# minutes_num distribution by delay status
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["is_delayed"] == label]["minutes_num"]
    axes[0, 0].hist(subset, bins=30, alpha=0.6, label=f"{'Delayed' if label else 'On-time'}", color=color, density=True)
axes[0, 0].set_xlabel("Minutes to Arrival")
axes[0, 0].set_title("Minutes-to-Arrival Distribution")
axes[0, 0].legend()

# num_trains_at_station
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["is_delayed"] == label]["num_trains_at_station"]
    axes[0, 1].hist(subset, bins=20, alpha=0.6, label=f"{'Delayed' if label else 'On-time'}", color=color, density=True)
axes[0, 1].set_xlabel("Trains at Station")
axes[0, 1].set_title("Trains at Station Distribution")
axes[0, 1].legend()

# Rolling delay rate (station)
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["is_delayed"] == label]["delay_rate_30min"]
    axes[1, 0].hist(subset, bins=30, alpha=0.6, label=f"{'Delayed' if label else 'On-time'}", color=color, density=True)
axes[1, 0].set_xlabel("Station Delay Rate (30-min rolling)")
axes[1, 0].set_title("Rolling Station Delay Rate")
axes[1, 0].legend()

# Rolling delay rate (line)
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["is_delayed"] == label]["line_delay_rate_30min"]
    axes[1, 1].hist(subset, bins=30, alpha=0.6, label=f"{'Delayed' if label else 'On-time'}", color=color, density=True)
axes[1, 1].set_xlabel("Line Delay Rate (30-min rolling)")
axes[1, 1].set_title("Rolling Line Delay Rate")
axes[1, 1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_realtime_features.png"), dpi=150)
plt.show()

## 6. Incident Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Delay rate: with vs without active incident
inc_groups = df.groupby("active_incident")["is_delayed"].mean()
inc_labels = ["No Incident", "Active Incident"]
axes[0].bar(inc_labels, inc_groups.values, color=["#78909C", "#E53935"])
for i, v in enumerate(inc_groups.values):
    axes[0].text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")
axes[0].set_ylabel("Delay Rate")
axes[0].set_title("Delay Rate: Incident vs No Incident")

# Delay rate: incident type = Delay specifically
inc_type = df.groupby("incident_is_delay")["is_delayed"].mean()
type_labels = ["No Delay Incident", "Active Delay Incident"]
axes[1].bar(type_labels, inc_type.values, color=["#78909C", "#D32F2F"])
for i, v in enumerate(inc_type.values):
    axes[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")
axes[1].set_ylabel("Delay Rate")
axes[1].set_title("Impact of Delay-Type Incidents")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_incident_impact.png"), dpi=150)
plt.show()

print(f"Rows with active incident: {df['active_incident'].sum():,} ({df['active_incident'].mean():.1%})")
print(f"Rows with delay incident:  {df['incident_is_delay'].sum():,} ({df['incident_is_delay'].mean():.1%})")

## 7. Feature Correlations

In [ ]:
# Correlation of numeric features with target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = df[numeric_cols].corr()["is_delayed"].drop("is_delayed").sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of correlations with target
colors = ["#F44336" if v > 0 else "#2196F3" for v in corr_with_target.values]
corr_with_target.plot.barh(ax=axes[0], color=colors)
axes[0].set_xlabel("Pearson Correlation with is_delayed")
axes[0].set_title("Feature Correlations with Delay Label")
axes[0].axvline(0, color="black", lw=0.8)

# Full correlation heatmap (top features only)
top_features = corr_with_target.abs().nlargest(10).index.tolist() + ["is_delayed"]
corr_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=axes[1], square=True)
axes[1].set_title("Correlation Matrix (Top Features)")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "eda_correlations.png"), dpi=150)
plt.show()

## 8. Key Takeaways\n\n- **Class balance** is reasonable (~37% delayed) — no need for aggressive oversampling.\n- **Red Line** has the highest delay rate, consistent with it being the busiest/longest line.\n- **Temporal patterns** show clear variation by hour — mid-morning and late-night have elevated delay rates.\n- **Rolling features** (`delay_rate_30min`, `line_delay_rate_30min`) show strong separation between delayed/on-time, suggesting they will be powerful predictors.\n- **Incidents** correlate with higher delay rates, as expected.\n- **GTFS scheduled headway** has ~53% missing values (direction mapping mismatch) — will need to handle or drop for modeling.\n- **Next step:** Baseline model with logistic regression using temporal + line + rolling features.